# Audio Event Detection — Training Notebook (Colab)

This notebook trains the multi-label audio event detection model using data **already saved to Google Drive**.

**Prerequisites:** Run the **data setup notebook** (`colab_data_setup.ipynb`) first to download and prepare the data.

Training is **resumable** across Colab sessions — checkpoints are saved to Google Drive automatically.

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the project
import os
PROJECT_DIR = '/content/audio-event-detection'
DRIVE_DIR = '/content/drive/MyDrive/audio-event-detection'

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'logs'), exist_ok=True)

In [ ]:
# Install dependencies
!pip install -q torch torchaudio timm librosa soundfile scikit-learn pyyaml matplotlib tqdm

In [ ]:
import subprocess

if os.path.exists(PROJECT_DIR):
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ankur-paul/audio-event-detection.git", PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)

In [ ]:
import shutil

local_data = os.path.join(PROJECT_DIR, 'data')
drive_data = os.path.join(DRIVE_DIR, 'data')

# Verify key files exist
assert os.path.exists('data/labels.csv'), "labels.csv not found — run data setup notebook first"
assert os.path.exists('data/class_map.json'), "class_map.json not found — run data setup notebook first"
print("✓ Data files verified")

## 2. Configuration

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.utils.config import load_config

# Load config with Drive checkpoint dir for persistence
config = load_config(overrides={
    'training': {
        'epochs': 50,
        'batch_size': 32,
        'num_workers': 2,
    },
})

print(f'Config loaded. Device: {"cuda" if __import__("torch").cuda.is_available() else "cpu"}')

## 3. Prepare Data

In [ ]:
from src.utils.logger import setup_logger
from src.data.dataset_preparation import (
    create_class_map, load_class_map, parse_labels_csv, split_dataset
)
from src.data.dataset import create_dataloaders

logger = setup_logger(log_level='INFO', console=True)

# Create class map
class_map = create_class_map(save_path=config.paths.class_map_file)
config.model.num_classes = len(class_map)
print(f'{len(class_map)} sound classes')

# Load labels and split
entries = parse_labels_csv(config.paths.labels_csv)
train_entries, val_entries, test_entries = split_dataset(entries)

# Create data loaders
train_loader, val_loader = create_dataloaders(
    train_entries, val_entries, class_map, config
)
print(f'Train: {len(train_entries)}, Val: {len(val_entries)}, Test: {len(test_entries)}')

## 4. Build Model

In [ ]:
from src.models.audio_event_model import build_model

model = build_model(config)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## 5. Train (Resumable)

This cell will automatically resume from the latest checkpoint if one exists.
Checkpoints are saved to both local storage and Google Drive.

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
)

# Automatically resume if checkpoint exists
trainer.resume_training()

# Train
trainer.train()

## 6. Visualize Results

In [ ]:
from src.training.experiment_tracker import MetricsTracker
from src.visualization.visualizer import plot_training_curves

tracker = MetricsTracker(metrics_file=config.paths.metrics_file)
history = tracker.load_history()

fig = plot_training_curves(history)
fig.show()

## 7. Inference

In [ ]:
from src.inference.inference_pipeline import InferencePipeline
from src.training.checkpoint import CheckpointManager
from src.visualization.visualizer import plot_event_timeline

# Load best model
ckpt_mgr = CheckpointManager(checkpoint_dir=config.paths.checkpoint_dir)
ckpt_mgr.load_checkpoint(model=model, load_best=True)

# Create inference pipeline
pipeline = InferencePipeline(model=model, class_map=class_map, config=config)

# Run on a test file
# result = pipeline.predict_file('path/to/test_audio.wav')
# fig = plot_event_timeline(result)
# fig.show()